# 附录 A6：现代 DID 估计量的 Stata 实操入口

本 Notebook 配合第六章「双重差分法怎么选」使用。它的目标不是覆盖所有 DID 命令，而是展示几类关键问题：

- 交错处理数据长什么样；
- TWFE 为什么只能作为基准；
- `csdid` 如何围绕 $ATT(g,t)$ 工作；
- `drdid` 如何展示两期 DID 的双重稳健思路；
- 现代事件研究、Bacon 分解和 honest DiD 如何作为扩展检查。

所有数据均从在线地址读取，不在本地存储原始数据。若某些扩展命令安装失败，可以跳过对应模块，不影响本附录主线。


## A6.0 环境准备

本附录默认使用 Stata 17 或更高版本，Notebook kernel 可使用 `nbstata`。以下安装命令只需在首次运行时执行。为避免课堂环境中断，部分新命令使用 `capture` 包裹。


In [ ]:
clear all
set more off
set linesize 120
version 17

* 基础依赖
cap which ftools
if _rc ssc install ftools, replace

cap which reghdfe
if _rc ssc install reghdfe, replace

* DID 相关命令：若安装失败，记录错误后继续
cap which csdid
if _rc ssc install csdid, all replace

cap which drdid
if _rc ssc install drdid, replace

cap which bacondecomp
if _rc ssc install bacondecomp, replace

* 以下命令属于可选模块，安装源可能随时间变化
cap which eventstudyinteract
if _rc ssc install eventstudyinteract, replace

cap which did_imputation
if _rc ssc install did_imputation, replace

cap which did2s
if _rc ssc install did2s, replace


## A6.1 数据与处理时点结构

本附录使用 `mpdta.dta`。它常用于演示多期 DID 和交错处理设定。这里的核心变量包括：

- `countyreal`：县级单位；
- `year`：年份；
- `lemp`：就业结果变量；
- `lpop`：协变量；
- `first_treat`：首次接受处理的年份，0 通常表示 never-treated。

本节先不估计任何模型，只观察处理时间结构。


In [ ]:
use https://friosavila.github.io/playingwithstata/drdid/mpdta.dta, clear

describe
summarize lemp lpop first_treat year

tab year
tab first_treat

gen treated = first_treat > 0 & year >= first_treat
label var treated "Treatment status by first_treat and year"

tab year treated
xtset countyreal year


运行后重点看两点：

1. 是否存在 never-treated；
2. 不同处理组是否在不同年份首次接受处理。

这决定了本数据不是简单 2×2 DID，而是交错处理 DID。


## A6.2 TWFE 基准与 Bacon 分解诊断

先跑一个熟悉的 TWFE。这个结果可以作为基准，但不应在交错处理中直接作为最终答案。随后尝试 Bacon decomposition，查看 TWFE 背后的 2×2 比较来源。


In [ ]:
reghdfe lemp treated lpop, absorb(countyreal year) vce(cluster countyreal)
estimates store twfe_base

* Bacon decomposition：可选模块
capture noisily bacondecomp lemp treated, ddetail
if _rc {
    di as error "bacondecomp 未能运行。请检查命令安装或语法版本。"
}


解读重点：

- TWFE 系数把不同处理批次和不同时期的比较压缩成一个数；
- Bacon decomposition 的作用是诊断比较来源，而不是自动修正 TWFE；
- 如果 already-treated 被当作对照，处理效应随时间变化时就可能污染估计。


## A6.3 `csdid`：估计 $ATT(g,t)$ 并聚合

`csdid` 的关键不是命令本身，而是先估计 group-time ATT，再根据研究问题聚合。

这里分别展示 simple ATT、event aggregation，并尝试使用 not-yet-treated 对照组。


In [ ]:
* simple ATT
csdid lemp lpop, ivar(countyreal) time(year) ///
    gvar(first_treat) method(dripw) agg(simple)

estimates store csdid_simple

* event / dynamic ATT
csdid lemp lpop, ivar(countyreal) time(year) ///
    gvar(first_treat) method(dripw) agg(event)

estimates store csdid_event

* 使用 not-yet-treated 作为对照组：需结合研究设计判断是否合理
capture noisily csdid lemp lpop, ivar(countyreal) time(year) ///
    gvar(first_treat) method(dripw) agg(event) notyet

if _rc {
    di as error "notyet 版本未能运行。请检查当前 csdid 版本帮助文件。"
}


解读重点：

- `gvar(first_treat)` 表示首次处理时间，不是普通处理状态变量；
- `agg(simple)` 回答总体平均效应；
- `agg(event)` 回答处理后第 $k$ 年的动态效应；
- `notyet` 允许当前尚未处理的单位作为对照，但不能把已经处理的单位当作无政策反事实。


## A6.4 `drdid`：两期 DID 中的双重稳健示例

`drdid` 更适合两期 DID 或较简单 DID。为使用同一份在线数据，本节把 `mpdta` 压缩成一个 2×2 子样本：选择 2004 年首次处理的单位和 never-treated 单位，并保留 2003 与 2004 两年。


In [ ]:
use https://friosavila.github.io/playingwithstata/drdid/mpdta.dta, clear

keep if first_treat == 2004 | first_treat == 0
keep if inlist(year, 2003, 2004)

gen treat = first_treat == 2004
gen post  = year == 2004

tab year treat

* 普通 DID 基准
reg lemp i.treat##i.post lpop, vce(cluster countyreal)

* DRDID：语法随版本可能略有差异
capture noisily drdid lemp lpop, ivar(countyreal) time(year) tr(treat)

if _rc {
    di as error "drdid 未能运行。请在本地执行 help drdid 核验当前版本语法。"
}


解读重点：

- `drdid` 不是交错 DID 的主工具；
- 它适合展示两期 DID 中 outcome regression + IPW 的双重稳健思想；
- 当平行趋势需要在协变量条件下才更可信时，DRDID 是重要入口。


## A6.5 现代事件研究修正：可选模块

本节提供 `eventstudyinteract` 与 `did_imputation` 的示意入口。不同 Stata 版本、命令版本和数据格式可能需要调整。若运行失败，不影响前面主模块。


In [ ]:
use https://friosavila.github.io/playingwithstata/drdid/mpdta.dta, clear

gen treated = first_treat > 0 & year >= first_treat
gen rel_year = year - first_treat if first_treat > 0
gen never_treat = first_treat == 0

* 构造有限个 lead / lag
forvalues k = -4/5 {
    local nm = cond(`k' < 0, "m" + string(abs(`k')), "p" + string(`k'))
    gen rel_`nm' = rel_year == `k'
    replace rel_`nm' = 0 if missing(rel_`nm')
}

* 删除基准期 -1
drop rel_m1

* 传统 TWFE event-study 基准
reghdfe lemp rel_m4 rel_m3 rel_m2 rel_p0 rel_p1 rel_p2 rel_p3 rel_p4 rel_p5 lpop, ///
    absorb(countyreal year) vce(cluster countyreal)

* Sun-Abraham eventstudyinteract：可选
capture noisily eventstudyinteract lemp rel_m4 rel_m3 rel_m2 rel_p0 rel_p1 rel_p2 rel_p3 rel_p4 rel_p5, ///
    absorb(countyreal year) cohort(first_treat) control_cohort(never_treat) vce(cluster countyreal)

if _rc {
    di as error "eventstudyinteract 未能运行。请检查命令安装和语法版本。"
}

* BJS did_imputation：可选
capture noisily did_imputation lemp countyreal year first_treat, ///
    horizons(0/5) pretrend(4) controls(lpop) cluster(countyreal)

if _rc {
    di as error "did_imputation 未能运行。请检查命令安装和语法版本。"
}


解读重点：

- 传统 TWFE event-study 是直观入口，但在交错处理下可能有 lead / lag 污染；
- `eventstudyinteract` 从 cohort-specific event-time effects 角度修正；
- `did_imputation` 从先预测未处理反事实再计算效应的思路出发；
- 三者的目标参数和权重不同，结果不必完全相同。


## A6.6 honest DiD：平行趋势敏感性分析

本节为可选模块。honest DiD 的思想不是证明平行趋势成立，而是问：平行趋势允许偏离到什么程度，结论才会翻转？

Stata 版本可参考：

- https://github.com/mcaceresb/stata-honestdid

由于 `honestdid` 通常需要已有事件研究估计的系数和协方差矩阵，本节只放运行提醒。正式演示可在本地根据前面 event-study 输出整理后执行。


In [ ]:
* 可选安装方式，请以 GitHub README 为准：
* net install honestdid, from("https://raw.githubusercontent.com/mcaceresb/stata-honestdid/main") replace

* 典型流程：
* 1. 先运行事件研究，保存系数向量和协方差矩阵；
* 2. 指定政策前和政策后相对期；
* 3. 设置允许的趋势偏离范围；
* 4. 使用 honestdid 输出敏感性区间。

di as text "honestdid 模块为可选敏感性分析。请根据本地安装说明运行。"


## A6.7 多估计量同框对比图

正式写第六章时，可以在本地把 TWFE event-study、`csdid, agg(event)`、`eventstudyinteract` 和 `did_imputation` 的动态效应整理成同一张图。课堂中不需要要求学生全部复现，但这张图有助于说明：不同 DID 估计量可能回答相近但不完全相同的问题。


In [ ]:
* 占位模块：
* 运行完前面各估计量后，可将 event-time、coef、se、method 整理为长表，
* 再用 twoway connected / rcap 绘制多估计量同框图。
*
* 建议输出变量结构：
* event_time  coef  se  method
*
* twoway ///
*   (rcap ub lb event_time if method=="csdid") ///
*   (connected coef event_time if method=="csdid") ///
*   (connected coef event_time if method=="twfe") ///
*   , xline(-1) yline(0)

di as text "多估计量同框图需在本地整理结果后生成。"


## A6.8 方法与软件地图

正文中不要把所有命令讲成教程。课后实操建议优先查看以下资源：

- Asjad Naqvi, DiD 方法与软件地图：<https://asjadnaqvi.github.io/DiD/>
- Pedro Sant’Anna, DiD Resources：<https://psantanna.com/did-resources/>
- Rios-Avila, `stpackages`：<https://github.com/friosavila/stpackages>
- `csdid` GitHub 目录：<https://github.com/friosavila/stpackages/tree/main/csdid>
- `drdid` GitHub 目录：<https://github.com/friosavila/stpackages/tree/main/drdid>
- Compare-DiD-Estimators：<https://github.com/friosavila/Compare-DiD-Estimators>
- Stata `honestdid`：<https://github.com/mcaceresb/stata-honestdid>
